# 05 — Visualizations

Publication-quality figures using outputs from notebooks 03–04 / `main.py`.
Figures saved to `results/figures/`.

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

from src.visualization import (
    plot_anomaly_timeline,
    plot_spectrogram,
    plot_umap_latent_space,
    plot_score_distributions,
    plot_roc_curves,
    plot_shap_summary,
    save_figure,
)
from src.spectral_features import build_standard_timeseries
from src.pipeline_helpers import build_ensemble_models, ensemble_weights_from_config
from src.models import EnsembleDetector
from nb_common import (
    NotebookSettings, load_project_config, load_dataframe,
    load_cached_splits, build_splits, scale_splits, load_notebook_artifacts, setup_notebook,
)

SETTINGS = NotebookSettings(use_sample=True, use_cache=True)
cfg = load_project_config(SETTINGS)
setup_notebook(cfg)
FIGURES_DIR = Path(cfg['visualization']['figure_dir'])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = cfg['visualization'].get('dpi', 150)

## 2. Load Results

In [2]:
artifacts = None
try:
    artifacts = load_notebook_artifacts()
    test_scores = dict(artifacts['test_scores'])
    val_scores = artifacts['val_scores']
    y_val = artifacts['y_val']
    y_test = artifacts['y_test']
    lab_test = artifacts['lab_test']
    test_ts = artifacts['test_timestamps']
    feat_names = list(artifacts['feat_names'])
    print('Loaded notebook artifacts')
except FileNotFoundError as e:
    print(f'WARNING: {e}')
    test_scores, val_scores, y_val, y_test, lab_test, test_ts, feat_names = {}, {}, None, None, None, None, []

splits = load_cached_splits(SETTINGS, cfg)
if splits is None:
    df_tmp = load_dataframe(SETTINGS, cfg)
    splits = build_splits(SETTINGS, cfg, df_tmp)
MODELS_DIR = SETTINGS.resolve_output_dir() / 'models'
_, _, X_test, _ = scale_splits(splits, MODELS_DIR, load_existing=True)
weight_map = ensemble_weights_from_config(cfg)
if not test_scores:
    from src.pipeline_helpers import score_all_models, build_ensemble_models, train_detectors, calibrate_all_detectors
    y_test = splits['y_test']
    test_ts = splits['test_timestamps']
    lab_test = splits['lab_test']
    feat_names = list(splits['feat_names'])
if test_scores:
    ens = EnsembleDetector(models={k: None for k in weight_map}, weights=weight_map)
    if 'ensemble' not in test_scores:
        test_scores['ensemble'] = ens.score_from({k: test_scores[k] for k in weight_map if k in test_scores})
    if val_scores and 'ensemble' not in val_scores:
        val_scores = dict(val_scores)
        val_scores['ensemble'] = ens.score_from({k: val_scores[k] for k in weight_map if k in val_scores})
elif y_test is None:
    y_test = splits['y_test']
    test_ts = splits['test_timestamps']
    lab_test = splits['lab_test']

Loaded notebook artifacts


## 3. Anomaly Timeline

In [3]:
if 'ensemble' in test_scores and y_test is not None:
    if val_scores is None or y_val is None:
        val_scores = {'ensemble': test_scores['ensemble']}
        y_val = y_test
    opt_thr, _ = EnsembleDetector.find_optimal_threshold(
        val_scores['ensemble'], y_val, method='f1_optimal',
    )
    fig = plot_anomaly_timeline(test_ts, test_scores['ensemble'], y_test, threshold=opt_thr)
    save_figure(fig, 'anomaly_timeline', str(FIGURES_DIR), dpi=DPI)
    plt.show()

## 4. Spectrogram

In [4]:
df = load_dataframe(SETTINGS, cfg)
first_col = cfg['spectral']['input_series'][0]
ts_col = 'Timestamp' if 'Timestamp' in df.columns else 'Stime'
ts_df = build_standard_timeseries(df, timestamp_col=ts_col, bin_size_seconds=cfg['spectral']['bin_size_seconds'])
if first_col in ts_df.columns:
    fig = plot_spectrogram(ts_df[first_col].values, fs=1.0 / cfg['spectral']['bin_size_seconds'], title=f'Spectrogram — {first_col}')
    save_figure(fig, 'spectrogram', str(FIGURES_DIR), dpi=DPI)
    plt.show()

## 5. UMAP Latent Space

In [5]:
try:
    from src.models import VAETrainer
    vae = VAETrainer.load(str(MODELS_DIR / 'vae.pt'))
    latent = vae.vae.get_latent(X_test)
    fig = plot_umap_latent_space(latent, y_test, class_names=['BENIGN', 'ATTACK'])
    save_figure(fig, 'umap_latent', str(FIGURES_DIR), dpi=DPI)
    plt.show()
except Exception as exc:
    print(f'UMAP skipped: {exc}')

## 6. Score Distributions

In [6]:
if 'ensemble' in test_scores:
    fig = plot_score_distributions(test_scores['ensemble'], y_test, class_names=['BENIGN', 'ATTACK'])
    save_figure(fig, 'score_distributions', str(FIGURES_DIR), dpi=DPI)
    plt.show()

## 7. ROC Curves

In [7]:
if lab_test is not None and test_scores:
    fig = plot_roc_curves(lab_test, test_scores)
    save_figure(fig, 'roc_multimodel', str(FIGURES_DIR), dpi=DPI)
    plt.show()
else:
    print('ROC curves skipped — need labels and scores')

## 8. SHAP (Isolation Forest)

In [8]:
try:
    from src.models import IsolationForestDetector
    if_model = IsolationForestDetector.load(str(MODELS_DIR / 'if.pkl'))
    fig = plot_shap_summary(if_model.model, X_test[: min(500, len(X_test))], feat_names)
    save_figure(fig, 'shap_summary', str(FIGURES_DIR), dpi=DPI)
    plt.show()
except Exception as exc:
    print(f'SHAP skipped: {exc}')

## 9. Summary

In [9]:
saved = list(FIGURES_DIR.glob('*.png'))
print(f'Saved {len(saved)} figures to {FIGURES_DIR}')
for p in sorted(saved):
    print(' ', p.name)

Saved 6 figures to results\figures
  anomaly_timeline.png
  roc_multimodel.png
  score_distributions.png
  shap_summary.png
  spectrogram.png
  umap_latent.png
